# Проверка train_relabel_v1

Сверяем новую разметку с `train_dedup_v1`, исходным JSONL и журналом правок. Числа считаются на всех строках. В конце сохранено сравнение трёх прогонов char-LR.

In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import json

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data/processed/train_relabel_v1").is_dir())
folder = root / "data/processed/train_relabel_v1"
manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
reference = root / "data/test/test_gold.parquet"


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_csv(info):
    with (root / info["path"]).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


information = ("source", "raw_source", "source_removed_duplicates", "dataset", "label_changes", "affected_duplicates")
for name in information:
    info = manifest[name]
    assert sha256(root / info["path"]) == info["sha256"], name
for name, digest in manifest["build_files_sha256"].items():
    assert sha256(root / name) == digest, name
assert sha256(reference) == "aa2ead6a49182215e41416adf5ee84792bab6a36b4657744e86bcc5e08c65b76"

with (root / manifest["raw_source"]["path"]).open(encoding="utf-8") as stream:
    raw = [json.loads(line) for line in stream]
incoming = pq.read_table(root / manifest["source"]["path"])
outgoing = pq.read_table(root / manifest["dataset"]["path"])
assert incoming.schema == outgoing.schema
assert len(raw) == manifest["raw_source"]["records"]
assert len(incoming) == manifest["source"]["records"]
assert len(outgoing) == manifest["dataset"]["records"] == len(incoming)

changes = read_csv(manifest["label_changes"])
assert len(changes) == manifest["label_changes"]["records"]
print(f"Исходных записей {len(raw):,}. В каждой обработанной версии {len(incoming):,}.".replace(",", " "))
print("Хеши проверены, схема совпадает.")

Исходных записей 10 000. В каждой обработанной версии 4 749.
Хеши проверены, схема совпадает.


## Правки и происхождение

У каждой изменённой пары проверяем направление правки, цитату в описании и связь с исходным `item_id`.

In [2]:
def categories(row):
    return {entry["mcId"] for entry in row["found_microcategories"]}


before, after = incoming.to_pylist(), outgoing.to_pylist()
after_by_line = {row["source_line"]: row for row in after}
observed = {}
for old, new in zip(before, after, strict=True):
    assert all(old[key] == new[key] for key in old if key != "found_microcategories")
    assert raw[new["source_line"] - 1]["item_id"] == new["item_id"]
    old_labels, new_labels = categories(old), categories(new)
    assert len(new_labels) == len(new["found_microcategories"])
    assert new_labels <= set(range(102, 107))
    for cat in old_labels ^ new_labels:
        observed[new["source_line"], cat] = (int(cat in old_labels), int(cat in new_labels))

reported = {}
for change in changes:
    line, cat = int(change["source_line"]), int(change["mcId"])
    row = after_by_line[line]
    assert (line, cat) not in reported
    assert row["item_id"] == int(change["item_id"])
    assert change["quote"] in row["description"]
    reported[line, cat] = int(change["old"]), int(change["new"])
assert observed == reported

changed_lines = {line for line, _ in reported}
assert len(changed_lines) == manifest["changed_rows"]
copies = read_csv(manifest["affected_duplicates"])
source_copies = read_csv(manifest["source_removed_duplicates"])
assert len(copies) == manifest["affected_duplicates"]["records"]
assert copies == [row for row in source_copies if int(row["kept_source_line"]) in changed_lines]
for copy in copies:
    assert int(copy["item_id"]) == raw[int(copy["source_line"]) - 1]["item_id"]
    assert int(copy["kept_item_id"]) == after_by_line[int(copy["kept_source_line"])]["item_id"]

table = pd.DataFrame.from_dict(manifest["transformations"][0]["by_category"], orient="index")
display(table.rename(columns={"before":"Было", "removed":"Снято", "added":"Добавлено", "after":"Стало"}))
print(f"Изменено {len(changed_lines)} объявлений. Прослежено {len(copies)} удалённых копий.")

Изменено 734 объявлений. Прослежено 764 удалённых копий.


,Было,Снято,Добавлено,Стало
102,2423,548,79,1954
103,2294,1,116,2409
104,2252,3,88,2337
105,2236,1,94,2329
106,2591,22,112,2681


## Примеры

Номер строки считается с единицы в исходном JSONL.

In [3]:
examples = {
    176: {106},
    210: {103, 104, 105},
    2598: {103, 104, 105, 106},
    8050: set(),
    9389: {102, 104, 105},
    9478: {102, 103, 104, 105, 106},
}
for line, expected in examples.items():
    assert categories(after_by_line[line]) == expected

group = [row for row in after if "укладка ламината, плитки, кварцвинила" in row["description"].casefold()]
assert len(group) == 10 and all(103 in categories(row) for row in group)
before_by_line = {row["source_line"]: row for row in before}
cases = pd.DataFrame([{
    "Строка": line,
    "Было": sorted(categories(before_by_line[line])),
    "Стало": sorted(categories(after_by_line[line])),
    "Цитаты": " / ".join(change["quote"] for change in changes if int(change["source_line"]) == line),
} for line in examples])
display(cases)
print("Во всех 10 объявлениях с группой работ «Укладка ламината, плитки, кварцвинила» стоит 103.")

Во всех 10 объявлениях с группой работ «Укладка ламината, плитки, кварцвинила» стоит 103.


,Строка,Было,Стало,Цитаты
0,176,"[102, 106]",[106],шпаклёвка
1,210,[],"[103, 104, 105]","укладка плитки, монтаж оборудования / Демонтаж..."
2,2598,"[102, 104, 105, 106]","[103, 104, 105, 106]",выравнивание стен / Укладка плитки (на стены и...
3,8050,"[102, 106]",[],ШПАКЛЁВКА / Подготовка поверхности под поклейк...
4,9389,"[102, 103, 104, 105]","[102, 104, 105]",стены в ванной - под плитку
5,9478,[],"[102, 103, 104, 105, 106]","штукатурка стен по маякам / Укладка плитки, ке..."


## Сравнение на test_gold

Один классификатор, один порядок строк в каждом разбиении. В обучении 3 820 записей, в dev 926, в test 139. Стенд исключил три одинаковых описания, встречающихся в train и test, по своему ключу нормализации. Таблица показывает сохранённые результаты. 

Для повторения вызови `bench.py prepare --train <parquet> --test data/test/test_gold.parquet --assume-source 101 --out <каталог-данных>` и `bench.py run --model char-lr --data <каталог-данных> --out <каталог-прогона>` для каждого варианта. Скрипт находится в `research/experiments/`.

In [4]:
results = {'train_dedup_v1': {'train_parquet_sha256': '57d0704e82a31c0ce91ec74196652f328b3f79951a35368fe759e0a0b7247fd3', 'test_hash': '6a273fa728002da47995e84ab2ce2293e49cbbaf0d68365b5d925b43176632bb', 'micro_F1': 0.84884, 'precision': 0.85882, 'recall': 0.83908, 'exact_match': 0.73381, 'F1 102': 0.72727, 'F1 103': 0.8, 'F1 104': 0.90411, 'F1 105': 0.89744, 'F1 106': 0.89552}, 'train_labels_102_v1': {'train_parquet_sha256': 'a658f683972f58433a91ec83799f661b185e4da5007efa295d4a896aeb7dd5ae', 'test_hash': '6a273fa728002da47995e84ab2ce2293e49cbbaf0d68365b5d925b43176632bb', 'micro_F1': 0.87613, 'precision': 0.92357, 'recall': 0.83333, 'exact_match': 0.81295, 'F1 102': 0.86792, 'F1 103': 0.8, 'F1 104': 0.90411, 'F1 105': 0.89744, 'F1 106': 0.89552}, 'train_relabel_v1': {'train_parquet_sha256': '141ab8f8510779cef4cd0bb8e96914e8ed6841ec6ff99babc34af1ecbfd1e717', 'test_hash': '6a273fa728002da47995e84ab2ce2293e49cbbaf0d68365b5d925b43176632bb', 'micro_F1': 0.89286, 'precision': 0.92593, 'recall': 0.86207, 'exact_match': 0.84173, 'F1 102': 0.83636, 'F1 103': 0.85246, 'F1 104': 0.91892, 'F1 105': 0.925, 'F1 106': 0.90909}}
assert len({row["test_hash"] for row in results.values()}) == 1
for name, run in results.items():
    assert sha256(root / "data/processed" / name / "train.parquet") == run["train_parquet_sha256"], name
display(pd.DataFrame.from_dict(results, orient="index").drop(columns=["test_hash", "train_parquet_sha256"]))
print("SHA-256 скрипта стенда при сравнении: 04439ab03a9776e25b6e4a6ef0ccac76d860c0c91cec24980a5261b0b61daa7f")

SHA-256 скрипта стенда при сравнении: 04439ab03a9776e25b6e4a6ef0ccac76d860c0c91cec24980a5261b0b61daa7f


,micro_F1,precision,recall,exact_match,F1 102,F1 103,F1 104,F1 105,F1 106
train_dedup_v1,0.84884,0.85882,0.83908,0.73381,0.72727,0.80000,0.90411,0.89744,0.89552
train_labels_102_v1,0.87613,0.92357,0.83333,0.81295,0.86792,0.80000,0.90411,0.89744,0.89552
train_relabel_v1,0.89286,0.92593,0.86207,0.84173,0.83636,0.85246,0.91892,0.92500,0.90909
